In [ ]:
# Import necessary libraries
import pyarrow.feather as feather
import pandas as pd
import talib
import numpy as np
from pathlib import Path

# Define the Block class
class Block:
    def __init__(self, start_date, end_date, start_price, end_price, duration, data_segment):
        self.start_date = start_date
        self.end_date = end_date
        self.start_price = start_price
        self.end_price = end_price
        self.duration = duration
        self.data_segment = data_segment  # Store all the data points in this block
        self.direction = 'UP' if start_price < end_price else 'DOWN'
        self.features = {}

# Function to load cryptocurrency data
def load_crypto_data(data_file_path, start_date, end_date):
    """
    Load cryptocurrency data, filter by date range, and calculate TEMA and trend information.
    
    Parameters:
    data_file_path (str): Path to the Feather file containing the dataset.
    start_date (str): The start date for filtering the data in 'YYYY-MM-DD' format.
    end_date (str): The end date for filtering the data in 'YYYY-MM-DD' format.
    
    Returns:
    pd.DataFrame: DataFrame containing the filtered data with TEMA, trend, and trend change information.
    """
    # Load data from Feather file
    crypto_df = feather.read_feather(data_file_path)

    # Convert input dates to datetime format
    start_date = pd.to_datetime(start_date).tz_localize('UTC')
    end_date = pd.to_datetime(end_date).tz_localize('UTC')
    
    # Filter data between the start and end dates
    crypto_df = crypto_df[(crypto_df['date'] >= start_date) & (crypto_df['date'] <= end_date)]

    # Calculate the Triple Exponential Moving Average (TEMA) with a default period of 50
    tema_period = 50
    crypto_df['tema'] = talib.TEMA(crypto_df['close'], timeperiod=tema_period)

    # Determine the trend direction (UP, DOWN, STABLE)
    crypto_df['trend'] = np.where(crypto_df['tema'] > crypto_df['tema'].shift(1), 'UP',
                                  np.where(crypto_df['tema'] < crypto_df['tema'].shift(1), 'DOWN', 'STABLE'))

    # Identify significant trend changes (ignoring 'STABLE' transitions)
    crypto_df['is_trend_change'] = crypto_df['trend'] != crypto_df['trend'].shift(1)
    crypto_df['is_significant_trend_change'] = crypto_df['is_trend_change'] & (crypto_df['trend'] != 'STABLE')

    # Assign a unique group ID to each continuous trend segment
    crypto_df['group_id'] = crypto_df['is_significant_trend_change'].cumsum()

    return crypto_df

# Function to create blocks from the data
def create_blocks(crypto_df):
    """
    Create a list of Block objects from the DataFrame.
    """
    blocks = []

    # Group the data by 'group_id'
    grouped = crypto_df.groupby('group_id')

    # Iterate over each group and create a block
    for group_id, group_data in grouped:
        start_date = group_data['date'].iloc[0]
        end_date = group_data['date'].iloc[-1]
        start_price = group_data['close'].iloc[0]
        end_price = group_data['close'].iloc[-1]
        duration = len(group_data)

        # Create a block object that includes all the data points
        block = Block(
            start_date=start_date,
            end_date=end_date,
            start_price=start_price,
            end_price=end_price,
            duration=duration,
            data_segment=group_data  # Store the entire segment of data
        )

        # Add the block to the list
        blocks.append(block)

    return blocks

# Example usage (for debugging purposes):
data_file_path = '/allah/freqtrade/user_data/data/binance/futures/ETH_USDT_USDT-3m-futures.feather'
start_date = '2024-01-01'
end_date = '2024-08-27'

# Load the data
crypto_df = load_crypto_data(data_file_path, start_date, end_date)

# Create blocks
blocks = create_blocks(crypto_df)


In [ ]:
blocks[-1].data_segment

In [ ]:
# Define the feature extraction function
def extract_features(block, selected_features=None):
    """
    Extract features from the block data.

    Parameters:
    block (Block): The Block object containing the data segment.
    selected_features (list): A list of feature names to include in the final feature dictionary.

    Returns:
    dict: A dictionary containing the extracted features.
    """
    # Define a dictionary to hold all potential features
    all_features = {}

    # Calculate the percentage change in price
    price_change = (block.end_price - block.start_price) / block.start_price
    all_features['price_change'] = price_change

    # Calculate the percentage change in price for each day
    block.data_segment['price_change'] = block.data_segment['close'].pct_change()

    # Calculate the average daily price change
    avg_daily_price_change = block.data_segment['price_change'].mean()
    all_features['avg_daily_price_change'] = avg_daily_price_change

    # Calculate the standard deviation of daily price changes
    std_daily_price_change = block.data_segment['price_change'].std()
    all_features['std_daily_price_change'] = std_daily_price_change

    # Example of additional features
    # Calculate max and min price in the block
    max_price = block.data_segment['close'].max()
    min_price = block.data_segment['close'].min()
    all_features['max_price'] = max_price
    all_features['min_price'] = min_price

    # Calculate total volume traded in the block
    total_volume = block.data_segment['volume'].sum()
    all_features['total_volume'] = total_volume

    all_features['length'] = block.duration

    def tsfresh_features(data):
        from tsfresh.feature_extraction import extract_features
        from tsfresh.feature_extraction.settings import MinimalFCParameters

        # Extract features using the 'MinimalFCParameters' settings
        extracted_features = extract_features(data, column_id='date', default_fc_parameters=MinimalFCParameters())
        

        return extracted_features

    # Filter the features to include only those specified in 'selected_features'
    if selected_features is not None:
        features = {key: all_features[key] for key in selected_features if key in all_features}
    else:
        features = all_features  # If no filter is applied, return all features

    return features

# Apply feature extraction to each block
for block in blocks:
    block.features = extract_features(block, selected_features=None)

# Output the features of the first block to check
print(blocks[-1].features)


In [ ]:
blocks[-1].features

In [ ]:
# This cell is for Y generation. Y is the length of the next block
Y = [blocks[i].duration for i in range(len(blocks) - 1)]

# Generate X: sequences of features from the previous k blocks in correct order
k = 5
X = [[blocks[i - k + j].features for j in range(k)] for i in range(k, len(blocks) - 1)]

# Align Y with X
Y = Y[k:]
l = 3
# Y = 1 if the length of the next block is greater than l
Y = [1 if y > l else 0 for y in Y]
# Check lengths
len(X), len(Y)


In [ ]:
X[-3], Y[-3]

In [ ]:
# # This cell is for Plotting
# Y = []
# for i in range(len(blocks) - 1):
#     Y.append(blocks[i + 1].duration)

# # Ensure the lengths of X and Y match
# k = 5
# X = []

# # Loop through the list of blocks, starting from the kth block to len(blocks) - 1
# for i in range(k, len(blocks) - 1):
#     # Collect features from the previous k blocks
#     features = [blocks[i - j - 1].features for j in range(k)]
    
#     # Append the collected features to the X list
#     X.append(features)

# # Ensure Y starts from the kth block as well
# Y = Y[k:]

# # Display lengths to check if they match
# print(f'Length of X: {len(X)}, Length of Y: {len(Y)}')

# # Import necessary library for plotting
# import matplotlib.pyplot as plt

# # Initialize a dictionary to store the counts for each l value
# counts = {}

# # Iterate over l from 1 to 20
# for l in range(1, 21):
#     # Generate the binary Y values based on the current l
#     binary_Y = [1 if y > l else 0 for y in Y]
    
#     # Count the number of 1s and 0s
#     unique_values = set(binary_Y)
#     unique_counts = {val: binary_Y.count(val) for val in unique_values}
    
#     # Store the counts
#     counts[l] = unique_counts
    
#     # Print the count for each unique value in binary_Y
#     print(f'For l = {l}: {unique_counts}')

# # Plot the counts for visualization
# l_values = list(counts.keys())
# count_0 = [counts[l].get(0, 0) for l in l_values]
# count_1 = [counts[l].get(1, 0) for l in l_values]

# plt.figure(figsize=(10, 5))
# plt.plot(l_values, count_0, label='Count of Y = 0', marker='o')
# plt.plot(l_values, count_1, label='Count of Y = 1', marker='o')
# plt.xlabel('Threshold l')
# plt.ylabel('Count')
# plt.title('Distribution of Binary Y Values Based on Threshold l')
# plt.legend()
# plt.grid(True)
# plt.show()


# unique, counts = np.unique(Y, return_counts=True)
# dict(zip(unique, counts))

In [ ]:
# Import numpy for array manipulation
import numpy as np

# Example input X with feature extraction for LSTM model
# Convert each list of dictionaries in X to a list of numerical values (e.g., 'length' value)
X_numeric = []

for sequence in X:
    sequence_numeric = [list(block.values()) for block in sequence]
    X_numeric.append(sequence_numeric)

# Convert X_numeric to a numpy array
X_numeric = np.array(X_numeric)

# Reshape X to match the LSTM input shape: (samples, time steps, features)
# In this case, we have 1 feature ('length'), and the number of time steps is the length of each sequence
X_lstm = X_numeric.reshape((X_numeric.shape[0], X_numeric.shape[1], 1))

# Check the shape of the prepared data
print("Shape of X_lstm:", X_lstm.shape)  # Expected: (number of samples, sequence length, number of features)


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import TensorBoard
import datetime
import matplotlib.pyplot as plt

# Convert the processed X_lstm to numpy arrays
X = X_lstm
Y = np.array(Y)

print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape)

# Build and compile the LSTM model
model = models.Sequential()
model.add(layers.LSTM(units=50, input_shape=(X.shape[1], X.shape[2]), return_sequences=False))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Model summary
model.summary()

# TensorBoard callback
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

# Train the model
history = model.fit(X, Y, epochs=1200, batch_size=32, validation_split=0.2, callbacks=[tensorboard_callback])

# Plot training history
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.show()

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy Over Epochs')
plt.legend()
plt.show()
